In [ ]:
# Import necessary libraries
import os
from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec
from transformers import AutoTokenizer

# Set environment variables
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "/home/shenyl/hf/model/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/"
)

# Initialize LLM with steering vector capability
llm = LLM(
    model="/home/shenyl/hf/model/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/",
    enable_steer_vector=True,
    enforce_eager=True,
    tensor_parallel_size=1
)



In [ ]:
# Define the suffix for newline tokens in the tokenizer
target_suffix = "ĊĊ"  # "\n\n" is tokenized as "ĊĊ"

# Get complete tokenizer vocabulary
vocab = tokenizer.get_vocab()

# Find all tokens and their IDs that end with the target suffix
# These are the newline tokens we'll apply steering to
matching_tokens_ids = [
    token_id
    for token, token_id in vocab.items()
    if isinstance(token, str) and token.endswith(target_suffix)
]

# Configure steering spec for SEAL control
# The three vectors (execution, reflection, transition) are applied to
# newline tokens during generation
steering = SteeringSpec(
    vectors=[
        # Execution vector (positive scale to promote execution-like text)
        VectorSpec(
            source="execution_avg_vector.gguf",
            scale=0.5,                    # Positive scale promotes this behavior
            layers=[20],                  # Apply at layer 20
            algorithm="direct",           # Direct application
            normalize=False,              # Do not normalize vectors
            apply=ApplySpec(phases=["generation"], tokens=matching_tokens_ids),
        ),

        # Reflection vector (negative scale to suppress reflection)
        VectorSpec(
            source="reflection_avg_vector.gguf",
            scale=-0.5,                   # Negative scale suppresses this behavior
            layers=[20],
            algorithm="direct",
            normalize=False,
            apply=ApplySpec(phases=["generation"], tokens=matching_tokens_ids),
        ),

        # Transition vector (negative scale to suppress transitions)
        VectorSpec(
            source="transition_avg_vector.gguf",
            scale=-0.5,                   # Negative scale suppresses this behavior
            layers=[20],
            algorithm="direct",
            normalize=False,
            apply=ApplySpec(phases=["generation"], tokens=matching_tokens_ids),
        ),
    ],

    # Additional parameters
    debug=False,                # Don't output debug info
    conflict="sequential",      # Apply vectors in sequence
)


# MATH500

In [ ]:
import json
file_path = "/home/xhl/eval/my_eval/data/math500/test.jsonl"

problems = []
answers = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        problems.append(item["problem"])
        answers.append(item["answer"])

# 看看前两个
print("Problems:", problems[:2])
print("Answers:", answers[:2])


examples = ["Please reason step by step, and put your final answer within \\boxed{}.\nUser: " + prompt + "\nAssistant: <think>" for prompt in problems]


In [ ]:
# Generate response with SEAL steering
example_answers = llm.generate(
    examples, 
    SamplingParams(
        temperature=0,
        max_tokens=8192,
        skip_special_tokens=False,
    ), 
    steering=steering
)

In [ ]:
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig
outputs = [output.outputs[0].text for output in example_answers]
extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
results = []
for i, llm_output in enumerate(outputs):
    gold = parse(f"${answers[i]}$", extraction_config=extraction_target)
    answer = parse(llm_output, extraction_config=extraction_target)
    result = verify(gold, answer)
    results.append(result)
accuracy = sum(results) / len(results)
print(accuracy)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "/home/shenyl/hf/model/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/"
)
length = 0
for i in range(len(outputs)):
    length += len(tokenizer.tokenize(outputs[i], add_special_tokens=True))
print("Length: ", length/len(outputs))

# GSM8k

In [ ]:
import json
file_path = "/home/xhl/eval/my_eval/data/gsm8k/test.jsonl"

problems = []
answers = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        problems.append(item["question"])
        answers.append(item["answer"])

# 看看前两个
print("Problems:", problems[:2])
print("Answers:", answers[:2])


examples = ["Please reason step by step, and put your final answer within \\boxed{}.\nUser: " + prompt + "\nAssistant: <think>" for prompt in problems]


In [ ]:
example_answers = llm.generate(
    examples, 
    SamplingParams(
        temperature=0,
        max_tokens=8192,
        skip_special_tokens=False,
    ), 
    steering=steering
)

In [ ]:
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig
outputs = [output.outputs[0].text for output in example_answers]
extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
results = []
for i, llm_output in enumerate(outputs):
    gold = parse(f"${answers[i]}$", extraction_config=extraction_target)
    answer = parse(llm_output, extraction_config=extraction_target)
    result = verify(gold, answer)
    results.append(result)
accuracy = sum(results) / len(results)
print(accuracy)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "/home/shenyl/hf/model/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/"
)
length = 0
for i in range(len(outputs)):
    length += len(tokenizer.tokenize(outputs[i], add_special_tokens=True))
print("Length: ", length/len(outputs))